In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, VBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# RANDOM WALK
# ============================================================

# ============================================================
# PREGENERATED RANDOM NUMBERS
# ============================================================

np.random.seed(83)

M_max = 200
N_max = 2000

U = np.random.rand(M_max, N_max)

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#12388c;
    margin-bottom:8px;
">
Random Walk as the Accumulation of Independent Random Steps
</div>

<div style="margin-bottom:4px;">
<b>Random increments:</b> w[n] takes the values +A and −A with probabilities p and 1−p.
</div>

<div style="margin-bottom:4px;">
<b>Random walk:</b> x[n] is obtained by accumulating the independent increments, x[n] = Σw[k].
</div>

<div style="margin-bottom:4px;">
Its mean and variance depend on time; for p = 0.5, E{x[n]} = 0 and Var{x[n]} = A²n.
</div>

<div>
<b>This notebook:</b> compares individual random paths with their theoretical ensemble mean and variance.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='160px')

p_slider = FloatSlider(min=0.10, max=0.90, step=0.05, value=0.50, description=' ', readout=False, continuous_update=True, style=slider_style, layout=slider_layout)

A_slider = FloatSlider(min=0.5, max=3.0, step=0.5, value=1.0, description=' ', readout=False, continuous_update=True, style=slider_style, layout=slider_layout)

M_slider = IntSlider(min=20, max=200, step=20, value=100, description=' ', readout=False, continuous_update=True, style=slider_style, layout=slider_layout)

N_slider = IntSlider(min=100, max=2000, step=100, value=800, description=' ', readout=False, continuous_update=True, style=slider_style, layout=slider_layout)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

p_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.50</div>')

A_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.0</div>')

M_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">100</div>')

N_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">800</div>')

# ============================================================
# UPDATE CURRENT VALUES
# ============================================================

def update_p_value(change):
    p_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{p_slider.value:.2f}</div>'

def update_A_value(change):
    A_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{A_slider.value:.1f}</div>'

def update_M_value(change):
    M_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{M_slider.value}</div>'

def update_N_value(change):
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

p_slider.observe(update_p_value, names='value')
A_slider.observe(update_A_value, names='value')
M_slider.observe(update_M_value, names='value')
N_slider.observe(update_N_value, names='value')

# ============================================================
# CONTROL LABELS
# ============================================================

p_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Probability p:</div>')

A_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Step amplitude A:</div>')

M_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Realizations M:</div>')

N_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Steps N:</div>')

# ============================================================
# CONTROLS GRID
# ============================================================

controls_grid = GridBox(
    children=[
        p_label, p_slider, p_value,
        A_label, A_slider, A_value,
        M_label, M_slider, M_value,
        N_label, N_slider, N_value
    ],
    layout=Layout(
        width='790px',
        grid_template_columns='115px 160px 55px 125px 160px 55px',
        grid_template_rows='34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#12388c;
            margin-bottom:5px;
        ">
        Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='820px',
        padding='10px 14px',
        border='1px solid #d2d2d2',
        overflow='hidden',
        margin='10px 0px 10px 0px'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_random_walk(p=0.50, A=1.0, M=100, N=800):

    # --------------------------------------------------------
    # RANDOM STEPS
    # --------------------------------------------------------

    W = np.where(U[:M, :N] < p, A, -A)

    # --------------------------------------------------------
    # RANDOM WALK
    # --------------------------------------------------------

    X = np.cumsum(W, axis=1)

    n = np.arange(1, N + 1)

    # --------------------------------------------------------
    # ESTIMATED STATISTICS
    # --------------------------------------------------------

    estimated_mean = np.mean(X, axis=0)

    estimated_variance = np.var(X, axis=0)

    # --------------------------------------------------------
    # THEORETICAL STATISTICS
    # --------------------------------------------------------

    step_mean = A * (2.0 * p - 1.0)

    step_variance = 4.0 * A**2 * p * (1.0 - p)

    theoretical_mean = n * step_mean

    theoretical_variance = n * step_variance

    # --------------------------------------------------------
    # DISPLAY RANGE
    # --------------------------------------------------------

    show_N = min(300, N)

    show_index = np.arange(1, show_N + 1)

    # ========================================================
    # CONTROLLED AXIS LIMITS
    #
    # These limits do NOT depend on p or on the particular
    # random realization.
    # ========================================================

    increment_limit = 3.5

    walk_limit = 1.10 * A * show_N

    mean_limit = 1.10 * A * N

    variance_limit = 1.10 * A**2 * N

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(figsize=(10.4, 7.0))

    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.28)

    ax1 = fig.add_subplot(gs[0, 0])

    ax2 = fig.add_subplot(gs[0, 1])

    ax3 = fig.add_subplot(gs[1, 0])

    ax4 = fig.add_subplot(gs[1, 1])

    # ========================================================
    # GRAPH 1: RANDOM INCREMENTS
    # ========================================================

    ax1.step(show_index, W[0, :show_N], where='mid', linewidth=1.0)

    ax1.set_xlim(1, show_N)

    ax1.set_ylim(-increment_limit, increment_limit)

    ax1.set_xlabel('Time index n', fontsize=11)

    ax1.set_ylabel('w[n]', fontsize=11)

    ax1.set_title('Independent Random Steps', fontsize=13, pad=9)

    ax1.tick_params(axis='both', labelsize=9)

    ax1.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 2: ONE RANDOM WALK REALIZATION
    # ========================================================

    ax2.plot(show_index, X[0, :show_N], linewidth=1.2)

    ax2.set_xlim(1, show_N)

    ax2.set_ylim(-walk_limit, walk_limit)

    ax2.set_xlabel('Time index n', fontsize=11)

    ax2.set_ylabel('x[n]', fontsize=11)

    ax2.set_title('One Random-Walk Realization', fontsize=13, pad=9)

    ax2.tick_params(axis='both', labelsize=9)

    ax2.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 3: ENSEMBLE MEAN
    # ========================================================

    ax3.plot(n, estimated_mean, linewidth=1.6, label='Estimated ensemble mean')

    ax3.plot(n, theoretical_mean, linestyle='--', linewidth=2.0, label='Theoretical mean')

    ax3.set_xlim(1, N)

    ax3.set_ylim(-mean_limit, mean_limit)

    ax3.set_xlabel('Time index n', fontsize=11)

    ax3.set_ylabel('Mean', fontsize=11)

    ax3.set_title('Ensemble Mean versus Time', fontsize=13, pad=9)

    ax3.tick_params(axis='both', labelsize=9)

    ax3.grid(True, linestyle=':', alpha=0.5)

    ax3.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=1, fontsize=8)

    # ========================================================
    # GRAPH 4: ENSEMBLE VARIANCE
    # ========================================================

    ax4.plot(n, estimated_variance, linewidth=1.6, label='Estimated ensemble variance')

    ax4.plot(n, theoretical_variance, linestyle='--', linewidth=2.0, label='Theoretical variance')

    ax4.set_xlim(1, N)

    ax4.set_ylim(0, variance_limit)

    ax4.set_xlabel('Time index n', fontsize=11)

    ax4.set_ylabel('Variance', fontsize=11)

    ax4.set_title('Ensemble Variance versus Time', fontsize=13, pad=9)

    ax4.tick_params(axis='both', labelsize=9)

    ax4.grid(True, linestyle=':', alpha=0.5)

    ax4.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=1, fontsize=8)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(left=0.08, right=0.97, top=0.93, bottom=0.15)

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL RESULTS
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.42;
        width:920px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Mean of one increment:</b>
    E{{w[n]}} = A(2p−1) = {step_mean:.4f}

    &nbsp;&nbsp;&nbsp;

    <b>Variance of one increment:</b>
    Var{{w[n]}} = 4A²p(1−p) = {step_variance:.4f}

    <br>

    <b>At n = {N}:</b>
    theoretical E{{x[n]}} = {theoretical_mean[-1]:.4f}

    &nbsp;&nbsp;&nbsp;

    theoretical Var{{x[n]}} = {theoretical_variance[-1]:.4f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_random_walk,
    {
        'p': p_slider,
        'A': A_slider,
        'M': M_slider,
        'N': N_slider
    }
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1050px;
    padding:11px 15px;
    border:1px solid #c8dfce;
    background:#f8fcf9;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#197b35;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The random walk accumulates independent increments, so every new value contains the complete history of all previous steps.
</div>

<div style="margin-bottom:4px;">
For p = 0.5, the theoretical mean is zero while the variance grows linearly as Var{x[n]} = A²n.
</div>

<div>
Changing p introduces a positive or negative drift. The controlled axis scales make this change directly visible without automatic rescaling.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        controls_card,
        output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)